In [17]:
import sys
sys.path.append("../../../")

import NuLib.src.scripts.interpolate_table as it 
import h5py
import numexpr as ne

rho = 15e13 # g/ccm
T = 20 # MeV
Ye = 0.3
ig = 5 # group number
s = 0 # species
nulib_table_name = "/mnt/scratch/MRSN_crossing/NuLib/NuLib_rho82_temp65_ye51_ng18_ns3_version1.0_20250321.h5"
eos_table_name = "/mnt/scratch/tables/EOS/LS220_234r_136t_50y_analmu_20091212_SVNr26.h5"

nulib_table = h5py.File(nulib_table_name)
eos_table = h5py.File(eos_table_name)

print("rho =",rho,"g/ccm")
print("T =",T,"MeV")
print("Ye =",Ye)

# This attempt to interpolate a single point is failing due to trying to call np.where on a scalar:
# {
# 	"name": "ValueError",
# 	"message": "Calling nonzero on 0d arrays is not allowed. Use np.atleast_1d(scalar).nonzero() instead. If the context of this error is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.",
# 	"stack": "---------------------------------------------------------------------------
# ValueError                                Traceback (most recent call last)
# /tmp/ipykernel_3877427/3739640036.py in <module>
#      26     print()
#      27     print(\"Species\",s)
# ---> 28     print(\"Emissivity =\",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, \"emissivities\"),\"ergs/cm^3/s/MeV/srad\")
#      29     print(\"Absorption opacity =\",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, \"absorption_opacity\"),\"1/cm\")
#      30     print(\"Scattering opacity =\",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, \"scattering_opacity\"),\"1/cm\")

# /mnt/scratch/MRSN_crossing/SedonuGR/BH2_50ms/../../NuLib/src/scripts/interpolate_table.py in interpolate_eas(ig, s, rho, T, Ye, table, datasetname)
#     135     xt = np.array(table[\"ye_points\"])
#     136 
# --> 137     badlocs = np.where((rho<table[\"rho_points\"][0]) | (rho>table[\"rho_points\"][len(zt)-1]) \\
#     138        | (T<table[\"temp_points\"][0]) | (T>table[\"temp_points\"][len(yt)-1]) \\
#     139        | (Ye<table[\"ye_points\"][0]) | (Ye>table[\"ye_points\"][len(xt)-1]))

# ValueError: Calling nonzero on 0d arrays is not allowed. Use np.atleast_1d(scalar).nonzero() instead. If the context of this error is of the form `arr[nonzero(cond)]`, just use `arr[cond]`."
# }
# # print("Neutrino Energy:",nulib_table["neutrino_energies"][ig],"MeV")
# print("Energy bin size:", nulib_table["bin_widths"][ig],"MeV")
# for s in range(3):
#     print()
#     print("Species",s)
#     print("Emissivity =",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, "emissivities"),"ergs/cm^3/s/MeV/srad")
#     print("Absorption opacity =",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, "absorption_opacity"),"1/cm")
#     print("Scattering opacity =",it.interpolate_eas(ig, s, rho, T, Ye, nulib_table, "scattering_opacity"),"1/cm") 
    
mu_e = it.interpolate_eos(rho,T,Ye, eos_table, "mu_e")
mu_p = it.interpolate_eos(rho,T,Ye, eos_table, "mu_p")
mu_n = it.interpolate_eos(rho,T,Ye, eos_table, "mu_n")
mu_nue = mu_p + mu_e - mu_n
mu_nua = -1*(mu_nue)

print(f"Chemical potentials:\nmu_nue: ", mu_nue, "\nmu_nue_bar: ", mu_nua, "\nmu_p: ", mu_p, "\nmu_n: ", mu_n, "\nmu_e-: ", mu_e)
print(f"{mu_e} + {mu_p} = {mu_n} + {mu_nue}")

rho = 150000000000000.0 g/ccm
T = 20 MeV
Ye = 0.3
Chemical potentials:
mu_nue:  [136.65993328] 
mu_nue_bar:  [-136.65993328] 
mu_p:  [-59.43593563] 
mu_n:  [-19.8910641] 
mu_e-:  [176.20480481]
[176.20480481] + [-59.43593563] = [-19.8910641] + [136.65993328]
